# SevenNet relaxations on a Kaggle GPU

Runs `sevennet_worker.py` on a GPU; only the device changes compared with a CPU run. Needs: accelerator GPU (T4 or P100) and internet ON in the notebook settings; the files `sevennet_worker.py`, `input.extxyz` and (optionally, to resume) `output.json` added as a dataset. Resumable: rows already in `output.json` are kept.

In [ ]:
%pip install -q sevenn==0.13.0
import torch, sevenn
print('sevenn', sevenn.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, json, os, shutil
src = glob.glob('/kaggle/input/**/input.extxyz', recursive=True)[0]
folder = os.path.dirname(src)
work = '/kaggle/working'
for f in ['sevennet_worker.py', 'input.extxyz', 'output.json']:
    if os.path.exists(os.path.join(folder, f)):
        shutil.copy(os.path.join(folder, f), work)
print('files from', folder)
if os.path.exists(os.path.join(work, 'output.json')):
    print('rows already done:', len(json.load(open(os.path.join(work, 'output.json')))['rows']))

In [ ]:
# the run; the first call downloads the 7net-mf-ompa checkpoint. Progress every 5 structures.
import os, subprocess, sys
env = dict(os.environ, MLIP_DEVICE='cuda')
p = subprocess.Popen([sys.executable, '/kaggle/working/sevennet_worker.py', 'relax',
                      '/kaggle/working/input.extxyz', '/kaggle/working/output.json'],
                     env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'logm result may be inaccurate' in line or line.startswith('  return f('):
        continue  # harmless scipy warning at every cell step
    print(line, end='')
print('exit code', p.wait())

In [ ]:
import json
out = json.load(open('/kaggle/working/output.json'))
rows = out['rows']
err = [r['id'] for r in rows if 'error' in r]
conv = sum(1 for r in rows if r.get('converged'))
print(len(rows), 'rows |', conv, 'converged |', len(err), 'errors', err[:5])
print('devices used:', out.get('devices_used'))
print('Download /kaggle/working/output.json from the Output panel.')